# Notebook 15 — stable evaluation and ViT-equivalent geometry

Purpose: cover research steps1–3: stabilize the representation readout; recover an exact ViT boundary; isolate geometry constraints; train one optional-spatial candidate against ViT. Both online and EMA encoders change within the full JEPA pipeline, while predictor, masks, objective and paired data remain common.

Inputs: source-verified Notebook14 final feature caches; the original10,100 Food101 training and2,020 reused development images; four fresh paired seeds. Outputs: converged regularized linear probes, kNN20 checks, training-only routing diagnostics,8models,16initial/final readouts, figures and `handoff_15_to_16.json`. Test predictions remain closed.

**Run All performs the actual experiment by default**, launching a detached coordinator and two GPU workers. It returns launch/status information; training continues independently of your connection. Expected runtime: approximately1–3hours, subject to measured preflight and probe convergence; hard cap3hours. The validation profile takes a few minutes and contains software fixtures only.

Checkpoints: `checkpoints/<RUN_ID>/stages/worker0/arms/` and `worker1/arms/`, every50updates, epoch/stage boundary and at least every15minutes. Probe checkpoints store the complete convex optimizer and RNG state. Latest two valid states and best training-objective state are retained.

**Exact resume:** leave RUN_ID, configuration and exported source unchanged; Restart Kernel and Run All before the original deadline. The notebook validates checkpoint hashes, restored steps and original clock. An expired clock does not restart. A completed run displays saved results without new training.

The optional geometry has raw64D query/key coordinates and a signed spatial bias whose coefficient begins at zero. Content temperature and spatial length are independent and fixed in the default one-factor experiment. A signed bias is not a guaranteed metric or manifold isomorphism. Four seed pairs provide developmental screening, not final confirmation. Notebook16 is run separately.

**Saved validation outputs:** any software-fixture outputs below come from short build checks. The full scientific study has not been launched during notebook authoring. Default Run All still selects `full`.


**Operational recovery, 9 October 2026:** the first full worker launch exposed a stale evaluation-helper name before scientific training. The original failure is retained. The current `vit-geometry-001` run uses a validated, source-bound in-memory adapter and retains the original configuration, seeds, checkpoint identities and 19:55 UTC deadline. For exact resume of this run, use [the recovery launch notebook](../experiments/15_operational_recovery/launch.ipynb) in a fresh `ahr-go` kernel. Its runtime imports only tagged definition cells. The original executable notebook cells remain frozen for source verification.

## 1. Load reusable definitions
Only cells explicitly tagged `ahr-export` are imported; no upstream experiment cells execute.

In [ ]:
from pathlib import Path
import os, json, types
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG",":4096:8")
ROOT=Path(os.environ.get("FMM_JEPA_ROOT","/home/benjamin/workspace/ahr_go/fmm_jeppa")).resolve()
_boot={"__file__":str(ROOT/"fmm_jepa/bootstrap.ipynb")}
for _cell in json.loads((ROOT/"fmm_jepa/bootstrap.ipynb").read_text())["cells"]:
    if _cell["cell_type"]=="code" and "ahr-export" in _cell.get("metadata",{}).get("tags",[]):
        exec(compile("".join(_cell["source"]),"bootstrap#ahr-export","exec"),_boot)
_boot["install_notebook_package"](ROOT)
from fmm_jepa.bootstrap import TaggedNotebookLoader
WORKFLOW=types.ModuleType("fmm15_workflow");WORKFLOW.__file__=str(ROOT/"experiments/15_vit_geometry/workflow.ipynb")
TaggedNotebookLoader(WORKFLOW.__file__).exec_module(WORKFLOW)


## 2. Choose the registered study
Default `full` starts real work. To execute only the software validation during authoring, set `FMM15_PROFILE=validation` and use a separate run ID before launching. Keep the same values when resuming.

In [ ]:
PROFILE=os.environ.get("FMM15_PROFILE","full")
RUN_ID=os.environ.get("FMM15_RUN_ID","vit-geometry-001" if PROFILE=="full" else "vit-geometry-validation")
CONFIG=WORKFLOW.study_configuration(RUN_ID,PROFILE)
CONFIG


## 3. Launch or resume
Coordinator: `fmm15-<RUN_ID>`; GPU workers append `-worker0` and `-worker1`. Attach: `tmux attach -t fmm15-<RUN_ID>`; detach: Ctrl-b then d. Logs: `artifacts/notebook_15/<RUN_ID>/`. An independent watchdog enforces the original three-hour deadline.

In [ ]:
RESULT=WORKFLOW.launch_study(ROOT,CONFIG)
RESULT

## 4. Inspect progress and saved results
While running, the saved heartbeat identifies the active phase. After completion, this cell shows the screening decision and charts. A failed gate or cutoff is reported explicitly; no longer run starts automatically.

In [ ]:
from IPython.display import display, Markdown, Image
STATUS=WORKFLOW.inspect_study(ROOT,RUN_ID)
display(STATUS)
REPORT=STATUS.get("report",RESULT if PROFILE=="validation" else {})
if REPORT.get("fixture_source"):
    display(Markdown("**SOFTWARE VALIDATION ONLY — no new scientific result.**"))
for figure in REPORT.get("figures",[]):
    display(Image(filename=figure))


## Next decision
A positive handoff requires all registered endpoints, certified probe convergence, useful gain over initialization and acceptable paired quality. It permits Notebook16 to test compression; it does not establish superiority. The closed official evaluation is reserved for the frozen confirmation protocol in Notebook16.

References: [Attention Is All You Need](https://papers.nips.cc/paper/7181-attention-is-all-you-need.pdf), [I-JEPA](https://openaccess.thecvf.com/content/CVPR2023/html/Assran_Self-Supervised_Learning_From_Images_With_a_Joint-Embedding_Predictive_Architecture_CVPR_2023_paper.html).
The converged readout uses the installed [PyTorch LBFGS solver](https://docs.pytorch.org/docs/main/generated/torch.optim.LBFGS.html), with complete optimizer state saved and training-only stopping criteria.
